In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman'],
    'mathtext.fontset':  'stix',
    'font.size':          11,
    'axes.labelsize':     12,
    'axes.titlesize':     12,
    'xtick.labelsize':    10,
    'ytick.labelsize':    10,
    'axes.facecolor':    'white',
    'figure.facecolor':  'white',
    'savefig.facecolor': 'white',
    'axes.linewidth':     0.8,
    'xtick.major.width':  0.8,
    'ytick.major.width':  0.8,
})

def sde_Euler_traj_extinction(n, t):
    h = t / n
    sqrt_h = np.sqrt(h)

    # Parameters satisfying R0E < 1
    mu    = 0.03;  alpha = 0.15
    ga1   = 0.15;  ga2   = 0.15
    bt1   = 0.5;  bt2   = 0.3   # reduced transmission rates
    rho1  = 0.5;   rho2  = 0.5
    sg1   = 0.3;   sg2   = 0.46

    # Verify R0E
    R0E = (mu*bt1*np.exp(sg1**2/(4*rho1)) / ((mu+alpha)*(mu+ga1))
         + alpha*mu*bt2*np.exp(sg2**2/(4*rho2))
           / ((mu+alpha)*(mu+ga2)*(mu+ga1)))
    print(f"R0E = {R0E:.4f}  ({'< 1, extinction expected' if R0E < 1 else '>= 1'})")

    S     = np.zeros(n); V     = np.zeros(n); I     = np.zeros(n)
    S_det = np.zeros(n); V_det = np.zeros(n); I_det = np.zeros(n)
    cc1   = np.zeros(n); cc2   = np.zeros(n)

    S[0] = S_det[0] = 0.35
    V[0] = V_det[0] = 0.25
    I[0] = I_det[0] = 0.30
    cc1[0] = np.log(bt1)
    cc2[0] = np.log(bt2)

    dW = np.random.normal(0, 1, (n, 2))

    for j in range(1, n):
        # OU update
        cc1[j] = (cc1[j-1] + rho1*(np.log(bt1) - cc1[j-1])*h
                  + sg1*dW[j, 0]*sqrt_h)
        cc2[j] = (cc2[j-1] + rho2*(np.log(bt2) - cc2[j-1])*h
                  + sg2*dW[j, 1]*sqrt_h)
        b1 = np.exp(cc1[j-1])
        b2 = np.exp(cc2[j-1])

        # Stochastic
        S[j] = max(S[j-1] + (mu - (mu+alpha)*S[j-1]
                              - b1*S[j-1]*I[j-1])*h, 0)
        V[j] = max(V[j-1] + (alpha*S[j-1] - b2*V[j-1]*I[j-1]
                              - (mu+ga2)*V[j-1])*h, 0)
        I[j] = max(I[j-1] + (b1*S[j-1]*I[j-1] + b2*V[j-1]*I[j-1]
                              - (mu+ga1)*I[j-1])*h, 0)

        # Deterministic
        S_det[j] = max(S_det[j-1] + (mu - (mu+alpha)*S_det[j-1]
                                      - bt1*S_det[j-1]*I_det[j-1])*h, 0)
        V_det[j] = max(V_det[j-1] + (alpha*S_det[j-1]
                                      - bt2*V_det[j-1]*I_det[j-1]
                                      - (mu+ga2)*V_det[j-1])*h, 0)
        I_det[j] = max(I_det[j-1] + (bt1*S_det[j-1]*I_det[j-1]
                                      + bt2*V_det[j-1]*I_det[j-1]
                                      - (mu+ga1)*I_det[j-1])*h, 0)

    return (S, V, I), (S_det, V_det, I_det)


def plot_extinction():
    np.random.seed(42)
    n, T  = 100000, 300
    t_arr = np.linspace(0, T, n)

    (S, V, I), (S_det, V_det, I_det) = sde_Euler_traj_extinction(n, T)

    labels_sto  = [r'$S(t)$', r'$V(t)$', r'$I(t)$']
    labels_det  = [r'$S^0$',  r'$V^0$',  r'$I^0$']
    colors_sto  = ['steelblue', 'seagreen', 'firebrick']
    colors_det  = ['steelblue', 'seagreen', 'firebrick']
    data_sto    = [S,     V,     I    ]
    data_det    = [S_det, V_det, I_det]

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

    for ax, ys, yd, ls, ld, col in zip(
            axes, data_sto, data_det,
            labels_sto, labels_det, colors_sto):
        ax.plot(t_arr, ys, color=col, lw=0.8,  alpha=0.85, label=ls)
        ax.plot(t_arr, yd, color=col, lw=1.4,
                linestyle='--', alpha=0.9,      label=ld + ' (det.)')
        ax.set_xlabel(r'Time $t$')
        ax.set_ylabel(ls)
        ax.tick_params(direction='in', which='both')
        ax.legend(fontsize=9)

    # Annotate R0E
    R0E = (0.03*0.30*np.exp(0.3**2/(4*0.5))/(0.18*0.15)
         + 0.15*0.03*0.20*np.exp(0.46**2/(4*0.5))/(0.18*0.15*0.15))
    #fig.suptitle(
    #    rf'Disease extinction: $\mathcal{{R}}_0^E = {R0E:.3f} < 1$',
    #    fontsize=12
    #)

    fig.tight_layout()
    fig.savefig('extinction_simulation.pdf',
                format='pdf', bbox_inches='tight')
    plt.show()


if __name__ == '__main__':
    plot_extinction()

In [2]:
import numpy; print("numpy:", numpy.__version__)
import scipy; print("scipy:", scipy.__version__)
import matplotlib; print("matplotlib:", matplotlib.__version__)
import pandas; print("pandas:", pandas.__version__)
import numba; print("numba:", numba.__version__)
import torch; print("torch:", torch.__version__)
import statsmodels; print("statsmodels:", statsmodels.__version__)

numpy: 1.20.1
scipy: 1.6.2
matplotlib: 3.3.4
pandas: 1.2.4
numba: 0.53.1
torch: 1.13.1
statsmodels: 0.12.2
